# Лабораторная работа 4

Tensorflow 2.x

1) Подготовка данных

2) Использование Keras Model API

3) Использование Keras Sequential + Functional API

https://www.tensorflow.org/tutorials

Для выполнения лабораторной работы необходимо установить tensorflow версии 2.0 или выше .

Рекомендуется использовать возможности Colab'а по обучению моделей на GPU.



In [6]:
import os
import tensorflow as tf
import numpy as np
import math
import timeit
import matplotlib.pyplot as plt

%matplotlib inline

In [7]:
device = '/GPU:0' if tf.config.list_physical_devices('GPU') else '/CPU:0'
print(device)

/CPU:0


# Подготовка данных
Загрузите набор данных из предыдущей лабораторной работы. 

In [8]:
def load_cifar10(num_training=49000, num_validation=1000, num_test=10000):
    """
    Fetch the CIFAR-10 dataset from the web and perform preprocessing to prepare
    it for the two-layer neural net classifier. These are the same steps as
    we used for the SVM, but condensed to a single function.
    """
    # Load the raw CIFAR-10 dataset and use appropriate data types and shapes
    cifar10 = tf.keras.datasets.cifar10.load_data()
    (X_train, y_train), (X_test, y_test) = cifar10
    X_train = np.asarray(X_train, dtype=np.float32)
    y_train = np.asarray(y_train, dtype=np.int32).flatten()
    X_test = np.asarray(X_test, dtype=np.float32)
    y_test = np.asarray(y_test, dtype=np.int32).flatten()

    # Subsample the data
    mask = range(num_training, num_training + num_validation)
    X_val = X_train[mask]
    y_val = y_train[mask]
    mask = range(num_training)
    X_train = X_train[mask]
    y_train = y_train[mask]
    mask = range(num_test)
    X_test = X_test[mask]
    y_test = y_test[mask]

    # Normalize the data: subtract the mean pixel and divide by std
    mean_pixel = X_train.mean(axis=(0, 1, 2), keepdims=True)
    std_pixel = X_train.std(axis=(0, 1, 2), keepdims=True)
    X_train = (X_train - mean_pixel) / std_pixel
    X_val = (X_val - mean_pixel) / std_pixel
    X_test = (X_test - mean_pixel) / std_pixel

    return X_train, y_train, X_val, y_val, X_test, y_test

# If there are errors with SSL downloading involving self-signed certificates,
# it may be that your Python version was recently installed on the current machine.
# See: https://github.com/tensorflow/tensorflow/issues/10779
# To fix, run the command: /Applications/Python\ 3.7/Install\ Certificates.command
#   ...replacing paths as necessary.

# Invoke the above function to get our data.
NHW = (0, 1, 2)
X_train, y_train, X_val, y_val, X_test, y_test = load_cifar10()
print('Train data shape: ', X_train.shape)
print('Train labels shape: ', y_train.shape, y_train.dtype)
print('Validation data shape: ', X_val.shape)
print('Validation labels shape: ', y_val.shape)
print('Test data shape: ', X_test.shape)
print('Test labels shape: ', y_test.shape)

/usr/local/python/3.12.1/lib/python3.12/site-packages/keras/src/datasets/cifar.py:18: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  d = cPickle.load(f, encoding="bytes")


Train data shape:  (49000, 32, 32, 3)
Train labels shape:  (49000,) int32
Validation data shape:  (1000, 32, 32, 3)
Validation labels shape:  (1000,)
Test data shape:  (10000, 32, 32, 3)
Test labels shape:  (10000,)


In [9]:
class Dataset(object):
    def __init__(self, X, y, batch_size, shuffle=False):
        """
        Construct a Dataset object to iterate over data X and labels y
        
        Inputs:
        - X: Numpy array of data, of any shape
        - y: Numpy array of labels, of any shape but with y.shape[0] == X.shape[0]
        - batch_size: Integer giving number of elements per minibatch
        - shuffle: (optional) Boolean, whether to shuffle the data on each epoch
        """
        assert X.shape[0] == y.shape[0], 'Got different numbers of data and labels'
        self.X, self.y = X, y
        self.batch_size, self.shuffle = batch_size, shuffle

    def __iter__(self):
        N, B = self.X.shape[0], self.batch_size
        idxs = np.arange(N)
        if self.shuffle:
            np.random.shuffle(idxs)
        return iter((self.X[i:i+B], self.y[i:i+B]) for i in range(0, N, B))


train_dset = Dataset(X_train, y_train, batch_size=64, shuffle=True)
val_dset = Dataset(X_val, y_val, batch_size=64, shuffle=False)
test_dset = Dataset(X_test, y_test, batch_size=64)

In [10]:
# We can iterate through a dataset like this:
for t, (x, y) in enumerate(train_dset):
    print(t, x.shape, y.shape)
    if t > 5: break

0 (64, 32, 32, 3) (64,)
1 (64, 32, 32, 3) (64,)
2 (64, 32, 32, 3) (64,)
3 (64, 32, 32, 3) (64,)
4 (64, 32, 32, 3) (64,)
5 (64, 32, 32, 3) (64,)
6 (64, 32, 32, 3) (64,)


#  Keras Model Subclassing API


Для реализации собственной модели с помощью Keras Model Subclassing API необходимо выполнить следующие шаги:

1) Определить новый класс, который является наследником tf.keras.Model.

2) В методе __init__() определить все необходимые слои из модуля tf.keras.layer

3) Реализовать прямой проход в методе call() на основе слоев, объявленных в __init__()

Ниже приведен пример использования keras API для определения двухслойной полносвязной сети. 

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras

In [11]:
class TwoLayerFC(tf.keras.Model):
    def __init__(self, hidden_size, num_classes):
        super(TwoLayerFC, self).__init__()        
        initializer = tf.initializers.VarianceScaling(scale=2.0)
        self.fc1 = tf.keras.layers.Dense(hidden_size, activation='relu',
                                   kernel_initializer=initializer)
        self.fc2 = tf.keras.layers.Dense(num_classes, activation='softmax',
                                   kernel_initializer=initializer)
        self.flatten = tf.keras.layers.Flatten()
    
    def call(self, x, training=False):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.fc2(x)
        return x


def test_TwoLayerFC():
    """ A small unit test to exercise the TwoLayerFC model above. """
    input_size, hidden_size, num_classes = 50, 42, 10
    x = tf.zeros((64, input_size))
    model = TwoLayerFC(hidden_size, num_classes)
    with tf.device(device):
        scores = model(x)
        print(scores.shape)
        
test_TwoLayerFC()

(64, 10)


Реализуйте трехслойную CNN для вашей задачи классификации. 

Архитектура сети:
    
1. Сверточный слой (5 x 5 kernels, zero-padding = 'same')
2. Функция активации ReLU 
3. Сверточный слой (3 x 3 kernels, zero-padding = 'same')
4. Функция активации ReLU 
5. Полносвязный слой 
6. Функция активации Softmax 

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Conv2D

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Dense

In [12]:
class ThreeLayerConvNet(tf.keras.Model):
    def __init__(self, channel_1, channel_2, num_classes):
        super(ThreeLayerConvNet, self).__init__()
        ########################################################################
        # TODO: Implement the __init__ method for a three-layer ConvNet. You   #
        # should instantiate layer objects to be used in the forward pass.     #
        ########################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        initializer = tf.initializers.VarianceScaling(scale=2.0)

        self.conv1 = tf.keras.layers.Conv2D(
            filters=channel_1,
            kernel_size=(5, 5),
            padding='same',
            activation='relu',
            kernel_initializer=initializer
        )

        self.conv2 = tf.keras.layers.Conv2D(
            filters=channel_2,
            kernel_size=(3, 3),
            padding='same',
            activation='relu',
            kernel_initializer=initializer
        )

        self.flatten = tf.keras.layers.Flatten()

        self.fc = tf.keras.layers.Dense(
            num_classes,
            activation='softmax',
            kernel_initializer=initializer
        )

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ########################################################################
        #                           END OF YOUR CODE                           #
        ########################################################################
        
    def call(self, x, training=False):
        scores = None
        ########################################################################
        # TODO: Implement the forward pass for a three-layer ConvNet. You      #
        # should use the layer objects defined in the __init__ method.         #
        ########################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        if len(x.shape) == 4 and x.shape[1] == 3:
            x = tf.transpose(x, [0, 2, 3, 1])

        x = self.conv1(x)
        x = self.conv2(x)
        x = self.flatten(x)
        scores = self.fc(x)

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ########################################################################
        #                           END OF YOUR CODE                           #
        ########################################################################        
        return scores

In [13]:
def test_ThreeLayerConvNet():    
    channel_1, channel_2, num_classes = 12, 8, 10
    model = ThreeLayerConvNet(channel_1, channel_2, num_classes)
    with tf.device(device):
        x = tf.zeros((64, 3, 32, 32))
        scores = model(x)
        print(scores.shape)

test_ThreeLayerConvNet()

(64, 10)


Пример реализации процесса обучения:

In [14]:
def train_part34(model_init_fn, optimizer_init_fn, num_epochs=1, is_training=False):
    """
    Simple training loop for use with models defined using tf.keras. It trains
    a model for one epoch on the CIFAR-10 training set and periodically checks
    accuracy on the CIFAR-10 validation set.
    
    Inputs:
    - model_init_fn: A function that takes no parameters; when called it
      constructs the model we want to train: model = model_init_fn()
    - optimizer_init_fn: A function which takes no parameters; when called it
      constructs the Optimizer object we will use to optimize the model:
      optimizer = optimizer_init_fn()
    - num_epochs: The number of epochs to train for
    
    Returns: Nothing, but prints progress during trainingn
    """    
    with tf.device(device):

        
        loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()
        
        model = model_init_fn()
        optimizer = optimizer_init_fn()
        
        train_loss = tf.keras.metrics.Mean(name='train_loss')
        train_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='train_accuracy')
    
        val_loss = tf.keras.metrics.Mean(name='val_loss')
        val_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='val_accuracy')
        
        t = 0
        for epoch in range(num_epochs):
            
            # Reset the metrics - https://www.tensorflow.org/alpha/guide/migration_guide#new-style_metrics
            train_loss.reset_state()
            train_accuracy.reset_state()
            
            for x_np, y_np in train_dset:
                with tf.GradientTape() as tape:
                    
                    # Use the model function to build the forward pass.
                    scores = model(x_np, training=is_training)
                    loss = loss_fn(y_np, scores)
      
                    gradients = tape.gradient(loss, model.trainable_variables)
                    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
                    
                    # Update the metrics
                    train_loss.update_state(loss)
                    train_accuracy.update_state(y_np, scores)
                    
                    if t % print_every == 0:
                        val_loss.reset_state()
                        val_accuracy.reset_state()
                        for test_x, test_y in val_dset:
                            # During validation at end of epoch, training set to False
                            prediction = model(test_x, training=False)
                            t_loss = loss_fn(test_y, prediction)

                            val_loss.update_state(t_loss)
                            val_accuracy.update_state(test_y, prediction)
                        
                        template = 'Iteration {}, Epoch {}, Loss: {}, Accuracy: {}, Val Loss: {}, Val Accuracy: {}'
                        print (template.format(t, epoch+1,
                                             train_loss.result(),
                                             train_accuracy.result()*100,
                                             val_loss.result(),
                                             val_accuracy.result()*100))
                    t += 1

In [15]:
hidden_size, num_classes = 4000, 10
learning_rate = 1e-2
print_every=100

def model_init_fn():
    return TwoLayerFC(hidden_size, num_classes)

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)

train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 3.2471542358398438, Accuracy: 4.6875, Val Loss: 2.868335485458374, Val Accuracy: 12.899999618530273
Iteration 100, Epoch 1, Loss: 2.266603708267212, Accuracy: 27.521657943725586, Val Loss: 1.881253957748413, Val Accuracy: 39.5
Iteration 200, Epoch 1, Loss: 2.0901730060577393, Accuracy: 31.48320960998535, Val Loss: 1.824425220489502, Val Accuracy: 40.29999923706055
Iteration 300, Epoch 1, Loss: 2.009631633758545, Accuracy: 33.430233001708984, Val Loss: 1.8940566778182983, Val Accuracy: 37.099998474121094
Iteration 400, Epoch 1, Loss: 1.9405899047851562, Accuracy: 35.41926574707031, Val Loss: 1.730743408203125, Val Accuracy: 40.900001525878906
Iteration 500, Epoch 1, Loss: 1.8952959775924683, Accuracy: 36.64546203613281, Val Loss: 1.6343915462493896, Val Accuracy: 43.5
Iteration 600, Epoch 1, Loss: 1.8642606735229492, Accuracy: 37.54679489135742, Val Loss: 1.6775649785995483, Val Accuracy: 42.79999923706055
Iteration 700, Epoch 1, Loss: 1.8377931118011475, Acc

Обучите трехслойную CNN. В tf.keras.optimizers.SGD укажите Nesterov momentum = 0.9 . 

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/optimizers/SGD

Значение accuracy на валидационной выборке после 1 эпохи обучения должно быть > 50% .

In [16]:
learning_rate = 3e-3
channel_1, channel_2, num_classes = 32, 16, 10
print_every=100

def model_init_fn():
    model = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    model = ThreeLayerConvNet(channel_1, channel_2, num_classes)

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return model

def optimizer_init_fn():
    optimizer = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    optimizer = tf.keras.optimizers.SGD(
        learning_rate=learning_rate,
        momentum=0.9,
        nesterov=True
    )

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return optimizer

train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 2.7213003635406494, Accuracy: 15.625, Val Loss: 7.439541339874268, Val Accuracy: 7.90000057220459
Iteration 100, Epoch 1, Loss: 2.1307873725891113, Accuracy: 27.537128448486328, Val Loss: 1.758040428161621, Val Accuracy: 37.79999923706055
Iteration 200, Epoch 1, Loss: 1.8909584283828735, Accuracy: 34.35945129394531, Val Loss: 1.572828769683838, Val Accuracy: 44.400001525878906
Iteration 300, Epoch 1, Loss: 1.773900032043457, Accuracy: 38.15406799316406, Val Loss: 1.4981663227081299, Val Accuracy: 48.20000076293945
Iteration 400, Epoch 1, Loss: 1.6934442520141602, Accuracy: 40.706825256347656, Val Loss: 1.4411972761154175, Val Accuracy: 48.400001525878906
Iteration 500, Epoch 1, Loss: 1.6359331607818604, Accuracy: 42.486900329589844, Val Loss: 1.4090831279754639, Val Accuracy: 50.19999694824219
Iteration 600, Epoch 1, Loss: 1.5987367630004883, Accuracy: 43.711002349853516, Val Loss: 1.3831442594528198, Val Accuracy: 51.099998474121094
Iteration 700, Epoch 1, 

# Использование Keras Sequential API для реализации последовательных моделей.

Пример для полносвязной сети:

In [17]:
learning_rate = 1e-2
print_every=100

def model_init_fn():
    input_shape = (32, 32, 3)
    hidden_layer_size, num_classes = 4000, 10
    initializer = tf.initializers.VarianceScaling(scale=2.0)
    layers = [
        tf.keras.layers.Flatten(input_shape=input_shape),
        tf.keras.layers.Dense(hidden_layer_size, activation='relu',
                              kernel_initializer=initializer),
        tf.keras.layers.Dense(num_classes, activation='softmax', 
                              kernel_initializer=initializer),
    ]
    model = tf.keras.Sequential(layers)
    return model

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate) 

train_part34(model_init_fn, optimizer_init_fn)

/usr/local/python/3.12.1/lib/python3.12/site-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Iteration 0, Epoch 1, Loss: 3.222038745880127, Accuracy: 10.9375, Val Loss: 2.969933032989502, Val Accuracy: 13.600000381469727
Iteration 100, Epoch 1, Loss: 2.226569414138794, Accuracy: 28.140470504760742, Val Loss: 1.9061591625213623, Val Accuracy: 38.10000228881836
Iteration 200, Epoch 1, Loss: 2.064340591430664, Accuracy: 32.1750602722168, Val Loss: 1.8314433097839355, Val Accuracy: 39.89999771118164
Iteration 300, Epoch 1, Loss: 1.9963074922561646, Accuracy: 34.068729400634766, Val Loss: 1.8828319311141968, Val Accuracy: 36.79999923706055
Iteration 400, Epoch 1, Loss: 1.92581307888031, Accuracy: 36.023223876953125, Val Loss: 1.7155685424804688, Val Accuracy: 41.20000076293945
Iteration 500, Epoch 1, Loss: 1.8855297565460205, Accuracy: 36.9261474609375, Val Loss: 1.6579291820526123, Val Accuracy: 42.69999694824219
Iteration 600, Epoch 1, Loss: 1.8549782037734985, Accuracy: 37.856178283691406, Val Loss: 1.699144721031189, Val Accuracy: 42.69999694824219
Iteration 700, Epoch 1, Loss:

Альтернативный менее гибкий способ обучения:

In [18]:
model = model_init_fn()
model.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate),
              loss='sparse_categorical_crossentropy',
              metrics=[tf.keras.metrics.sparse_categorical_accuracy])
model.fit(X_train, y_train, batch_size=64, epochs=1, validation_data=(X_val, y_val))
model.evaluate(X_test, y_test)

W0000 00:00:1777330131.876517   52892 cpu_allocator_impl.cc:82] Allocation of 602112000 exceeds 10% of free system memory.


766/766 ━━━━━━━━━━━━━━━━━━━━ 23s 30ms/step - loss: 1.8284 - sparse_categorical_accuracy: 0.3878 - val_loss: 1.6532 - val_sparse_categorical_accuracy: 0.4470
 22/313 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 1.6191 - sparse_categorical_accuracy: 0.4298

W0000 00:00:1777330173.734611   52892 cpu_allocator_impl.cc:82] Allocation of 122880000 exceeds 10% of free system memory.


313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - loss: 1.6421 - sparse_categorical_accuracy: 0.4424


[1.642137885093689, 0.4424000084400177]

Перепишите реализацию трехслойной CNN с помощью tf.keras.Sequential API . Обучите модель двумя способами.

In [19]:
def model_init_fn():
    model = None
    ############################################################################
    # TODO: Construct a three-layer ConvNet using tf.keras.Sequential.         #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    initializer = tf.initializers.VarianceScaling(scale=2.0)

    model = tf.keras.Sequential([
        tf.keras.layers.Conv2D(
            32,
            kernel_size=(5, 5),
            padding='same',
            activation='relu',
            kernel_initializer=initializer,
            input_shape=(32, 32, 3)
        ),
        tf.keras.layers.Conv2D(
            16,
            kernel_size=(3, 3),
            padding='same',
            activation='relu',
            kernel_initializer=initializer
        ),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(
            10,
            activation='softmax',
            kernel_initializer=initializer
        )
    ])

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                            END OF YOUR CODE                              #
    ############################################################################
    return model

learning_rate = 5e-4
print_every=100
def optimizer_init_fn():
    optimizer = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    optimizer = tf.keras.optimizers.SGD(
        learning_rate=learning_rate,
        momentum=0.9,
        nesterov=True
    )
  
    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return optimizer

train_part34(model_init_fn, optimizer_init_fn)

/usr/local/python/3.12.1/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Iteration 0, Epoch 1, Loss: 2.778688907623291, Accuracy: 14.0625, Val Loss: 2.523073196411133, Val Accuracy: 10.59999942779541
Iteration 100, Epoch 1, Loss: 2.0194804668426514, Accuracy: 28.573638916015625, Val Loss: 1.8188000917434692, Val Accuracy: 35.900001525878906
Iteration 200, Epoch 1, Loss: 1.8993204832077026, Accuracy: 33.29446792602539, Val Loss: 1.6894793510437012, Val Accuracy: 42.0
Iteration 300, Epoch 1, Loss: 1.8209761381149292, Accuracy: 35.91154479980469, Val Loss: 1.6424341201782227, Val Accuracy: 43.900001525878906
Iteration 400, Epoch 1, Loss: 1.7579163312911987, Accuracy: 38.0805778503418, Val Loss: 1.5832277536392212, Val Accuracy: 45.69999694824219
Iteration 500, Epoch 1, Loss: 1.7126944065093994, Accuracy: 39.70808410644531, Val Loss: 1.5210927724838257, Val Accuracy: 47.5
Iteration 600, Epoch 1, Loss: 1.6821058988571167, Accuracy: 40.84858703613281, Val Loss: 1.4721760749816895, Val Accuracy: 49.0
Iteration 700, Epoch 1, Loss: 1.6539167165756226, Accuracy: 41.8

In [20]:
model = model_init_fn()
model.compile(optimizer='sgd',
              loss='sparse_categorical_crossentropy',
              metrics=[tf.keras.metrics.sparse_categorical_accuracy])
model.fit(X_train, y_train, batch_size=64, epochs=1, validation_data=(X_val, y_val))
model.evaluate(X_test, y_test)

W0000 00:00:1777330243.127237   52892 cpu_allocator_impl.cc:82] Allocation of 602112000 exceeds 10% of free system memory.


766/766 ━━━━━━━━━━━━━━━━━━━━ 42s 54ms/step - loss: 1.5264 - sparse_categorical_accuracy: 0.4592 - val_loss: 1.3443 - val_sparse_categorical_accuracy: 0.5330
 24/313 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 1.2941 - sparse_categorical_accuracy: 0.5102

W0000 00:00:1777330285.700362   52892 cpu_allocator_impl.cc:82] Allocation of 122880000 exceeds 10% of free system memory.


313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 1.3362 - sparse_categorical_accuracy: 0.5230


[1.3361871242523193, 0.5230000019073486]

# Использование Keras Functional API

Для реализации более сложных архитектур сети с несколькими входами/выходами, повторным использованием слоев, "остаточными" связями (residual connections) необходимо явно указать входные и выходные тензоры. 

Ниже представлен пример для полносвязной сети. 

In [21]:
def two_layer_fc_functional(input_shape, hidden_size, num_classes):  
    initializer = tf.initializers.VarianceScaling(scale=2.0)
    inputs = tf.keras.Input(shape=input_shape)
    flattened_inputs = tf.keras.layers.Flatten()(inputs)
    fc1_output = tf.keras.layers.Dense(hidden_size, activation='relu',
                                 kernel_initializer=initializer)(flattened_inputs)
    scores = tf.keras.layers.Dense(num_classes, activation='softmax',
                             kernel_initializer=initializer)(fc1_output)

    # Instantiate the model given inputs and outputs.
    model = tf.keras.Model(inputs=inputs, outputs=scores)
    return model

def test_two_layer_fc_functional():
    """ A small unit test to exercise the TwoLayerFC model above. """
    input_size, hidden_size, num_classes = 50, 42, 10
    input_shape = (50,)
    
    x = tf.zeros((64, input_size))
    model = two_layer_fc_functional(input_shape, hidden_size, num_classes)
    
    with tf.device(device):
        scores = model(x)
        print(scores.shape)
        
test_two_layer_fc_functional()

(64, 10)


In [22]:
input_shape = (32, 32, 3)
hidden_size, num_classes = 4000, 10
learning_rate = 1e-2
print_every=100

def model_init_fn():
    return two_layer_fc_functional(input_shape, hidden_size, num_classes)

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)

train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 2.94571590423584, Accuracy: 12.5, Val Loss: 2.865274429321289, Val Accuracy: 11.59999942779541
Iteration 100, Epoch 1, Loss: 2.227997303009033, Accuracy: 28.589109420776367, Val Loss: 1.8634757995605469, Val Accuracy: 39.5
Iteration 200, Epoch 1, Loss: 2.073807716369629, Accuracy: 32.40049743652344, Val Loss: 1.919569492340088, Val Accuracy: 37.599998474121094
Iteration 300, Epoch 1, Loss: 1.9992583990097046, Accuracy: 34.30751419067383, Val Loss: 1.864335298538208, Val Accuracy: 36.89999771118164
Iteration 400, Epoch 1, Loss: 1.932986855506897, Accuracy: 35.988155364990234, Val Loss: 1.7409721612930298, Val Accuracy: 42.0
Iteration 500, Epoch 1, Loss: 1.8892691135406494, Accuracy: 37.085205078125, Val Loss: 1.6658920049667358, Val Accuracy: 42.599998474121094
Iteration 600, Epoch 1, Loss: 1.8591625690460205, Accuracy: 37.95497131347656, Val Loss: 1.6832250356674194, Val Accuracy: 42.79999923706055
Iteration 700, Epoch 1, Loss: 1.8336197137832642, Accuracy: 

Поэкспериментируйте с архитектурой сверточной сети. Для вашего набора данных вам необходимо получить как минимум 70% accuracy на валидационной выборке за 10 эпох обучения. Опишите все эксперименты и сделайте выводы (без выполнения данного пункта работы приниматься не будут). 

Эспериментируйте с архитектурой, гиперпараметрами, функцией потерь, регуляризацией, методом оптимизации.  

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/BatchNormalization#methods https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Dropout#methods

In [23]:
class CustomConvNet(tf.keras.Model):
    def __init__(self):
        super(CustomConvNet, self).__init__()
        ############################################################################
        # TODO: Construct a model that performs well on CIFAR-10                   #
        ############################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        initializer = tf.initializers.VarianceScaling(scale=2.0)

        self.conv1 = tf.keras.layers.Conv2D(32, (3, 3), padding='same',
                                            kernel_initializer=initializer)
        self.bn1 = tf.keras.layers.BatchNormalization()

        self.conv2 = tf.keras.layers.Conv2D(32, (3, 3), padding='same',
                                            kernel_initializer=initializer)
        self.bn2 = tf.keras.layers.BatchNormalization()
        self.pool1 = tf.keras.layers.MaxPooling2D(pool_size=(2, 2))
        self.drop1 = tf.keras.layers.Dropout(0.2)

        self.conv3 = tf.keras.layers.Conv2D(64, (3, 3), padding='same',
                                            kernel_initializer=initializer)
        self.bn3 = tf.keras.layers.BatchNormalization()

        self.conv4 = tf.keras.layers.Conv2D(64, (3, 3), padding='same',
                                            kernel_initializer=initializer)
        self.bn4 = tf.keras.layers.BatchNormalization()
        self.pool2 = tf.keras.layers.MaxPooling2D(pool_size=(2, 2))
        self.drop2 = tf.keras.layers.Dropout(0.3)

        self.flatten = tf.keras.layers.Flatten()

        self.fc1 = tf.keras.layers.Dense(256, activation='relu',
                                         kernel_initializer=initializer)
        self.drop3 = tf.keras.layers.Dropout(0.4)

        self.fc2 = tf.keras.layers.Dense(10, activation='softmax',
                                         kernel_initializer=initializer)
        
        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ############################################################################
        #                            END OF YOUR CODE                              #
        ############################################################################
    
    def call(self, input_tensor, training=False):
        ############################################################################
        # TODO: Construct a model that performs well on CIFAR-10                   #
        ############################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        x = self.conv1(input_tensor)
        x = self.bn1(x, training=training)
        x = tf.nn.relu(x)

        x = self.conv2(x)
        x = self.bn2(x, training=training)
        x = tf.nn.relu(x)
        x = self.pool1(x)
        x = self.drop1(x, training=training)

        x = self.conv3(x)
        x = self.bn3(x, training=training)
        x = tf.nn.relu(x)

        x = self.conv4(x)
        x = self.bn4(x, training=training)
        x = tf.nn.relu(x)
        x = self.pool2(x)
        x = self.drop2(x, training=training)

        x = self.flatten(x)
        x = self.fc1(x)
        x = self.drop3(x, training=training)
        x = self.fc2(x)

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ############################################################################
        #                            END OF YOUR CODE                              #
        ############################################################################
        return x


print_every = 700
num_epochs = 10

model = CustomConvNet()

def model_init_fn():
    return CustomConvNet()

def optimizer_init_fn():
    learning_rate = 1e-3
    return tf.keras.optimizers.Adam(learning_rate) 

train_part34(model_init_fn, optimizer_init_fn, num_epochs=num_epochs, is_training=True)

Iteration 0, Epoch 1, Loss: 4.507272720336914, Accuracy: 7.8125, Val Loss: 6.691345691680908, Val Accuracy: 11.100000381469727
Iteration 700, Epoch 1, Loss: 1.7449350357055664, Accuracy: 35.248748779296875, Val Loss: 1.3459104299545288, Val Accuracy: 51.89999771118164
Iteration 1400, Epoch 2, Loss: 1.3750301599502563, Accuracy: 48.90747833251953, Val Loss: 1.222159504890442, Val Accuracy: 58.60000228881836
Iteration 2100, Epoch 3, Loss: 1.2216862440109253, Accuracy: 55.461883544921875, Val Loss: 1.0642679929733276, Val Accuracy: 64.0
Iteration 2800, Epoch 4, Loss: 1.1435210704803467, Accuracy: 58.281558990478516, Val Loss: 0.9584683179855347, Val Accuracy: 66.19999694824219
Iteration 3500, Epoch 5, Loss: 1.094090223312378, Accuracy: 60.147308349609375, Val Loss: 0.9369896054267883, Val Accuracy: 67.19999694824219
Iteration 4200, Epoch 6, Loss: 1.0523295402526855, Accuracy: 61.682952880859375, Val Loss: 0.8604561686515808, Val Accuracy: 70.30000305175781
Iteration 4900, Epoch 7, Loss: 1

Опишите все эксперименты, результаты. Сделайте выводы.

В ходе работы последовательно исследовалось влияние архитектуры нейронной сети, способа реализации и параметров обучения на качество классификации изображений CIFAR-10. 

В 1 эксперименте использовалась простая двухслойная полносвязная сеть с оптимизатором SGD и learning rate = 1e-2. После одной эпохи обучения модель показала низкое качество: итоговая точность на валидационной выборке составила около 44%. Полносвязная архитектура не учитывает пространственную структуру изображений, а также в медленной сходимости SGD без использования дополнительных техник ускорения.

Во 2 эксперименте полносвязная сеть была заменена на сверточную (ThreeLayerConvNet), а в оптимизатор SGD добавлены momentum и Nesterov ускорения. Это привело к заметному росту качества до 53% на валидации за 1 эпоху. Сверточные слои эффективно извлекают локальные признаки изображений, а momentum снижает колебания при обновлении весов и ускоряет сходимость, в то время как Nesterov позволяет более точно корректировать направление градиентного спуска.

В 3 эксперименте была использована та же полносвязная архитектура, но реализованная через API tf.keras.Sequential. Точность на валидационной выборке достигла примерно 44%, результат совпадает с предыдущим для той же архитектуры, что подтверждает отсутствие влияния способа реализации (Sequential vs кастомная модель) на качество обучения, если архитектура и параметры обучения остаются прежними. 

В 4 эксперименте была реализована сверточная нейронная сеть (CNN) с использованием tf.keras.Sequential и обучена с применением оптимизатора SGD и ускорений momentum и Nesterov. В результате точность на валидационной выборке достигла 50%. Это выше, чем у полносвязной модели, что объясняется способностью свёрточных слоёв извлекать пространственные признаки изображений. Однако прирост оказался ограниченным, так как архитектура сети остаётся достаточно простой (отсутствуют BatchNorm, Dropout, увеличение глубины сети). Дополнительно модель была обучена через стандартный интерфейс model.compile() и model.fit(). Полученные результаты (test accuracy около 52–53%) сопоставимы с обучением через train_part34, что показывает, что способ обучения (кастомный цикл или встроенный Keras API) не оказывает существенного влияния на итоговое качество при одинаковых параметрах.

В 5 эксперименте была реализована более сложная сверточная сеть (CustomConvNet), включающая несколько сверточных слоев, Batch Normalization, функции активации ReLU, слои подвыборки (Pooling) и Dropout, оптимизатора Adam, 10 эпох обучения. В результате качество модели существенно выросло, и точность достигла 73% на валидационной выборке. Batch Normalization стабилизирует распределение активаций и ускоряет обучение, Dropout снижает переобучение, а Adam автоматически подбирает эффективный шаг обучения для каждого параметра, увеличение числа эпох также позволило модели лучше адаптироваться к данным.

Таким образом, результаты экспериментов показывают, что ключевыми факторами повышения качества являются переход к сверточным архитектурам, использование методов ускорения обучения (momentum, Adam) и применение регуляризации.
